# Model Comparison

**AST 384C · guest lecture classwork.** Fill in every `...` / `# TODO`, run the cell, then answer the ⏸ prompt in the markdown cell below it *before* moving on. Prediction-before-reveal is the point: write your guess down even if you're unsure.

**Toy problem:** $n$ Gaussian measurements with known $\sigma$. *Is the mean zero?*

| Part | Question it answers |
|---|---|
| 1 · Likelihood ratio test | Is the improvement in best fit more than chance would give under the simpler model? |
| 2 · Wilks' theorem | When can we read the LRT off a $\chi^2$ table, and when can't we? |
| 3 · Bayes factor | Which model predicted the data better, given its prior? |

Everything here is either closed form or a one-dimensional integral on a grid, so nothing needs a sampler. Only `numpy`, `scipy` and `matplotlib`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

%matplotlib inline
BLUE, ORANGE, AQUA, MUTED, INK = "#2a78d6", "#eb6834", "#1baf7a", "#898781", "#0b0b0b"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e1e0d9", "lines.linewidth": 2, "legend.frameon": False})

## Noisy dataset

We will generate a noisy dataset from a Gaussian distribution with one unknown parameter, its mean $\theta$:

$$x_i \sim \mathcal N(\theta, \sigma^2), \qquad \sigma = 1 \text{ known}.$$

- **M0 (null):** $\theta = 0$
- **M1:** $\theta$ free

Everything the data say about $\theta$ is contained in the sample mean $\bar x$ (it is a *sufficient statistic*), which has standard error $SE = \sigma/\sqrt n$.

Keep $n = 25$ for now. Part 2.3 asks you to change it.

In [ ]:
rngB = np.random.default_rng(1)
sigma = 1.0
n = 25                    # keep n = 25 until Part 2.3
theta_true = 0.4          # the true generating value: pretend we don't know it
xB = rngB.normal(theta_true, sigma, n)
SE = sigma / np.sqrt(n)
xbar = xB.mean()
print(f"xbar = {xbar:.3f}   SE = {SE:.3f}   z = xbar/SE = {xbar/SE:.2f}")

## Part 1: Likelihood ratio test

The LRT compares **nested** models: the simpler model is the general one with a parameter fixed at a specific value (here M0 is M1 with $\theta$ fixed at $\theta_0 = 0$). We use the Maximum Likelihood Estimator (MLE) to find the best-fit $\hat\theta$ in the larger model, and compare the two maximized **log**-likelihoods $\ell$:

$$\Lambda = 2\,\big[\,\ell(\hat\theta) - \ell(\theta_0)\,\big], \qquad \ell(\theta) = \log \mathcal L(\theta).$$

$\Lambda \ge 0$ always, because the larger model can always do at least as well as the smaller one.

**Task 1:** Write down the log-likelihood for a 1-D Gaussian model with known $\sigma$. This is analytically tractable and a useful exercise!

**Task 2:** Write down the LRT statistic function.

In [ ]:
def loglike(theta, x, sigma):
    """Gaussian log-likelihood of the data x given mean theta (sigma known)."""
    # TODO
    ...

def lrt_statistic(x, sigma):
    """Lambda = 2 * [ loglike(theta_hat) - loglike(0) ], with theta_hat the MLE (unconstrained)."""
    theta_hat = ...        # TODO: the MLE of a Gaussian mean
    return ...             # TODO

Lambda_obs = lrt_statistic(xB, sigma)
print(f"Lambda = {Lambda_obs:.3f}")

**Task 3:** Once you have the LRT statistic, you can use standard functions in `scipy.stats` to turn it into a p-value. If Wilks' theorem holds (Part 2), $\Lambda \sim \chi^2_k$ under M0, with $k$ the number of **extra** free parameters in M1. Use `stats.chi2.sf`: look at its docstring, and decide what `df` should be here.

In [ ]:
p_naive = ...          # TODO: stats.chi2.sf(?, df=?)
print(f"Lambda = {Lambda_obs:.3f}    naive p (chi2_1) = {p_naive:.4f}")
print(f"check: z^2 = {(xbar/SE)**2:.3f}   (for a Gaussian mean, Lambda = z^2 exactly)")

**From a statistic to a decision.** $\Lambda$ on its own is a *test statistic*, and the p-value says how surprising it would be under M0. What turns this into model selection is the threshold $\alpha$ that you, the experimenter, choose on the p-value: reject M0 if $p < \alpha$.

To avoid cherry-picking or p-hacking, you must choose $\alpha$ *a priori*, before looking at the data.

## Part 2: Wilks' theorem

Adding a parameter can only raise $\Lambda$, so how does the LRT avoid always preferring the bigger model? Not by subtracting a penalty, but by **calibrating** $\Lambda$: it compares the observed value with the distribution of $\Lambda$ you would get if M0 were true and only noise were at work.

Wilks' theorem says that, under regularity conditions and for large $n$, that distribution is $\chi^2_k$, with $k$ the number of *extra* free parameters in M1. The conditions are:

1. the models are nested;
2. the null value $\theta_0$ lies in the **interior** of M1's parameter space, not on its boundary;
3. every parameter of M1 is identifiable (it actually changes the likelihood) when M0 is true;
4. the likelihood is smooth enough and $n$ is large enough.

When any condition fails, $\chi^2_k$ is the wrong reference distribution and the naive p-value is wrong.

### 2.1 Does Wilks' theorem hold here? Check by simulation.

We can build the reference distribution ourselves: simulate many experiments in which M0 is true, compute $\Lambda$ for each, and compare their histogram with $\chi^2_1$. This is the frequentist picture in action: probability as a long-run frequency over repeated experiments. Wilks' theorem is a claim about *this distribution*, not about our one dataset.

In [ ]:
def simulate_null_Lambda(n_sims, n, sigma, rng, bounded=False):
    """Draw n_sims datasets of size n from M0 (theta = 0) and return their LRT statistics.
    If bounded=True, the MLE under M1 is constrained to theta >= 0."""
    out = np.empty(n_sims)
    for i in range(n_sims):
        x_sim = rng.normal(0.0, sigma, n)
        theta_hat = ...          # TODO: MLE; if bounded, clip at 0
        out[i] = ...             # TODO: Lambda for this dataset
    return out

rng = np.random.default_rng(10)
Lam_null = simulate_null_Lambda(20000, n, sigma, rng)

In [ ]:
# TODO: histogram of the simulated null Lam_null (density=True), overlay the chi2_1 pdf, and mark Lambda_obs
# HINT: stats.chi2.pdf(grid, 1) gives the chi2_1 curve; the empirical p-value is the fraction of Lam_null >= Lambda_obs
grid = np.linspace(0.02, 12, 400)
...

p_empirical = ...        # TODO
print(f"empirical p = {p_empirical:.4f}   vs naive p = {p_naive:.4f}")

### 2.2 Breaking Wilks: a physical boundary

Suppose $\theta$ is an amplitude that **cannot be negative**: an emission-line strength, a variance, a mass, a number of components. Now M1 is $\theta \ge 0$, and the null $\theta_0 = 0$ sits on the *edge* of M1's parameter space. That is condition 2 failing.

**⏸ Predict before running:** under the null, what fraction of simulated datasets will give $\Lambda$ *exactly* zero? Will the naive $\chi^2_1$ p-value be too big or too small?

*Your prediction:* ...

In [ ]:
Lam_bound = simulate_null_Lambda(20000, n, sigma, rng, bounded=True)

frac_zero = ...                       # TODO: fraction of Lam_bound that is exactly 0
p_corrected = ...                     # TODO: fraction of Lam_bound >= Lambda_obs

In [ ]:
# TODO: same plot as before for Lam_bound: its histogram, the naive chi2_1 pdf, the mixture distribution, and Lambda_obs.
# HINT: the mixture 1/2 delta_0 + 1/2 chi2_1 has two pieces. Draw its continuous part as 0.5 * stats.chi2.pdf(grid, 1),
#       and the spike at zero as a bar at Lambda = 0 of height frac_zero. Histogram only the draws with Lambda > 0.
...

print(f"fraction exactly zero = {frac_zero:.3f}")
print(f"naive p = {p_naive:.4f}    corrected p = {p_corrected:.4f}")

**⏸ Discuss.**
1. Why exactly that fraction at zero? What happened to the datasets whose sample mean came out negative?
2. The correct null is $\tfrac12\delta_0 + \tfrac12\chi^2_1$ (Self & Liang 1987). Is the naive p-value conservative or anti-conservative here?
3. Can you use an LRT to compare a Gaussian line profile against a Lorentzian one? Why or why not?
4. *(Stretch)* In a real spectrum you would also fit the line's centre and width. Why does that create a *second*, different problem for Wilks' theorem? Which condition fails?

*Your answers:* ...

### 2.3 What happens to the LRT as you collect more data?

Now keep the same true $\theta = 0.4$ and change $n$.

**⏸ Predict before running:** as $n$ grows from 10 to 10 000, what happens to (a) the observed $\Lambda$, (b) the p-value, and (c) the *null* distribution of $\Lambda$ that we compare it with?

*Your prediction:* ...

In [ ]:
rng_n = np.random.default_rng(2)
print(f"{'n':>6} {'xbar':>7} {'SE':>7} {'Lambda':>9} {'p (chi2_1)':>12}")
for n_ in [10, 25, 100, 1000, 10000]:
    x_ = rng_n.normal(theta_true, sigma, n_)
    Lam_ = ...                        # TODO: the LRT statistic for this dataset
    p_ = ...                          # TODO: its naive p-value
    print(f"{n_:6d} {x_.mean():7.3f} {sigma/np.sqrt(n_):7.3f} {Lam_:9.2f} {p_:12.2e}")

# the null distribution for a small and a large experiment
Lam_null_10 = simulate_null_Lambda(20000, 10, sigma, rng)
Lam_null_1000 = simulate_null_Lambda(20000, 1000, sigma, rng)
plt.hist(Lam_null_10, bins=np.linspace(0, 12, 61), density=True, histtype="step", color=BLUE, lw=2, label="null, n = 10")
plt.hist(Lam_null_1000, bins=np.linspace(0, 12, 61), density=True, histtype="step", color=ORANGE, lw=2, label="null, n = 1000")
plt.plot(grid, stats.chi2.pdf(grid, 1), color=INK, ls="--", label=r"$\chi^2_1$")
plt.ylim(0, 1.2); plt.xlabel("Λ"); plt.ylabel("density"); plt.legend(); plt.show()

**⏸ Discuss.**
1. Which part of the test depends on $n$: the observed $\Lambda$, or the reference distribution it is compared with?
2. Suppose the true effect were tiny, $\theta = 0.007$. Could the LRT still reject $\theta = 0$? What would you need? Would that detection be *important*?

*Your answers:* ...

## Part 3: Bayes factor

Computing a Bayes factor (BF) requires us to also define a **prior** on the parameters of each model. Unlike the LRT, the BF works for both nested and non-nested models. In this exercise, you decide the prior on $\theta$ under M1.

**Task 1:** Define a prior you want to test and plot its PDF. A uniform prior (a "box") is the easiest to build intuition with, but feel free to experiment with other forms!

Two rules for `prior(theta)`:
- it must be a **normalized** PDF (it integrates to 1 over $\theta$); a Task 2 cell checks this;
- it must return an array the same shape as `theta` (hint: `np.where`), not a single number.

Explore how the prior overlaps with the likelihood.

In [ ]:
def prior(theta):
    # TODO
    ...

th = np.linspace(-3, 3, 1200)
plt.plot(th, stats.norm.pdf(th, xbar, SE), color=BLUE, label="likelihood p(x̄ | θ)")
plt.plot(th, prior(th), color=ORANGE, label="prior p(θ)")
plt.axvline(xbar, color=INK, ls="--", lw=1, label="MLE (a single point)")
plt.axvline(0, color=MUTED, ls=":", label="null θ = 0")
plt.legend(); plt.xlabel("θ"); plt.ylabel("density"); plt.show()

Next we compute the evidence under each model:

$$Z_M = p(D \mid M) = \int p(D \mid \theta, M)\, p(\theta \mid M)\, d\theta$$

The evidence is the likelihood **averaged over the prior**: every value of $\theta$ the model allows gets a vote, weighted by how much prior probability the model put on it.

- **M0** puts all of its prior on $\theta = 0$ (a spike). The integral collapses to a single term: $Z_0 = p(D \mid \theta = 0)$.
- **M1** uses the prior you chose in Task 1. Compute $Z_1$ by integrating likelihood × prior on a grid of $\theta$.

Two practical notes:

1. We use $\bar x$ as the data, with $p(\bar x \mid \theta) = \mathcal N(\bar x;\, \theta,\, SE^2)$. Using $\bar x$ instead of all $n$ points changes both $Z_0$ and $Z_1$ by the same constant factor, so it cancels in the Bayes factor.
2. The likelihood is negligible more than a few $SE$ away from $\bar x$, so the grid only needs to cover that region. Only the part of the prior that sits under the likelihood contributes to $Z_1$. The rest of the prior is probability M1 "spent" on values the data ruled out.

The Bayes factor is the ratio of evidences, $BF_{10} = Z_1/Z_0$ (and $BF_{01} = 1/BF_{10}$). To put words on it we use the Kass & Raftery (1995) scale for whichever model the ratio favours: 1–3 *barely worth mentioning*, 3–20 *positive*, 20–150 *strong*, > 150 *very strong*.

**Task 2:** Compute the log evidence for both models, then the Bayes factor.

In [ ]:
def log_evidence_M0(xbar, se):
    """log p(xbar | M0): theta fixed at 0, so no integral is needed."""
    # TODO: the likelihood of xbar evaluated at theta = 0
    ...

def log_evidence_M1(xbar, se, prior, n_grid=20001):
    """log p(xbar | M1) = log ∫ p(xbar | theta) p(theta) dtheta, by numerical integration.
    `prior` is a function of theta. It must be a normalized PDF."""
    theta = np.linspace(xbar - 10 * se, xbar + 10 * se, n_grid)   # where the likelihood is non-negligible
    likelihood = ...          # TODO: p(xbar | theta) at every theta on the grid
    integrand = ...           # TODO: likelihood × prior
    return ...                # TODO: log of the integral (np.trapezoid)

def kass_raftery(bf01):
    """Words for a Bayes factor on the Kass & Raftery scale."""
    r = bf01 if bf01 >= 1 else 1 / bf01
    who = "M0 (θ = 0)" if bf01 >= 1 else "M1 (θ free)"
    if r < 3:   return "barely worth mentioning"
    if r < 20:  return f"positive evidence for {who}"
    if r < 150: return f"strong evidence for {who}"
    return f"very strong evidence for {who}"

# Check that your prior is a normalized PDF (widen the grid if your prior is wider than ±100)
g = np.linspace(-100, 100, 400001)
print(f"your prior integrates to {np.trapezoid(prior(g), g):.3f}   (should be ≈ 1)")

log_Z0 = log_evidence_M0(xbar, SE)
log_Z1 = log_evidence_M1(xbar, SE, prior)
log_BF10 = log_Z1 - log_Z0
print(f"Z0 = {np.exp(log_Z0):.4f}   Z1 = {np.exp(log_Z1):.4f}")
print(f"BF10 = {np.exp(log_BF10):.3f}   BF01 = {np.exp(-log_BF10):.3f}   ->  {kass_raftery(np.exp(-log_BF10))}")

**Task 3: same data, wider box.** The data never change from here on. Only the prior does. Use a uniform box of half-width $w$ and widen it.

**⏸ Predict before running:** as $w$ goes from 1 to 1000, does $BF_{01}$ go *up*, *down*, *up then down*, or *down then up*? Roughly where does it cross 1?

*Your prediction:* ...

In [ ]:
def uniform_prior(w, c=0.0):
    """Uniform on [c - w, c + w]."""
    return lambda theta: np.where(np.abs(theta - c) <= w, 1 / (2 * w), 0.0)

Z0 = np.exp(log_evidence_M0(xbar, SE))
print(f"{'w':>6} {'Z0':>8} {'Z1':>9} {'1/(2w)':>9} {'BF01':>9}   verdict")
for w in [1, 3, 10, 30, 100, 1000]:
    Z1_w = ...                              # TODO: evidence of M1 with uniform_prior(w)  (remember log_evidence_M1 returns a log)
    bf01_w = ...                            # TODO
    print(f"{w:6g} {Z0:8.4f} {Z1_w:9.4f} {1/(2*w):9.4f} {bf01_w:9.3f}   {kass_raftery(bf01_w)}")

# the summary curve
ws = np.logspace(-1, 3.5, 200)
bf01_curve = np.array([Z0 / np.exp(log_evidence_M1(xbar, SE, uniform_prior(w))) for w in ws])
w_flip = ...                                # TODO: the half-width where BF01 = 1, using Z1 ≈ 1/(2w) for wide boxes
plt.plot(ws, bf01_curve, color=AQUA, lw=2.5)
plt.axhline(1, color=INK, lw=1)
for lev, txt in [(3, "positive for M0"), (20, "strong for M0"), (150, "very strong for M0"), (1/3, "positive for M1"), (1/20, "strong for M1")]:
    plt.axhline(lev, color=MUTED, lw=0.6); plt.text(0.11, lev * 1.15, txt, color=MUTED, fontsize=9)
plt.axvline(w_flip, color=MUTED, ls="--"); plt.text(w_flip * 1.2, 300, f"flip at w ≈ {w_flip:.1f}", color=MUTED)
plt.xscale("log"); plt.yscale("log"); plt.ylim(0.03, 3000)
plt.xlabel("prior box half-width w"); plt.ylabel("BF₀₁  (evidence for θ = 0)"); plt.show()

**Task 4: the same boxes, seen by inference and by selection.** Under M1 the posterior for $\theta$ is the likelihood cut off at the box edges, $p(\theta \mid \bar x, M_1) \propto p(\bar x \mid \theta)\, p(\theta)$. Plot it for three box widths next to the three Bayes factors.

In [ ]:
th = np.linspace(xbar - 5 * SE, xbar + 5 * SE, 800)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
for w, col in [(1, "#f5b08f"), (10, ORANGE), (100, "#b34a1f")]:
    post = ...                              # TODO: likelihood × uniform_prior(w) on the grid th, normalized with np.trapezoid
    a1.plot(th, post, color=col, label=f"w = {w}")
    a2.plot([w], [Z0 / np.exp(log_evidence_M1(xbar, SE, uniform_prior(w)))], "o", color=col, ms=10, label=f"w = {w}")
a1.axvline(0, color=MUTED, ls=":"); a1.set_xlabel("θ"); a1.set_ylabel("posterior under M1")
a1.set_title("inference: p(θ | data, M1)"); a1.legend()
a2.plot(ws, bf01_curve, color=AQUA); a2.axhline(1, color=INK, lw=1); a2.set_xscale("log"); a2.set_yscale("log")
a2.set_xlabel("w"); a2.set_ylabel("BF₀₁"); a2.set_title("selection: BF₀₁"); a2.legend(); plt.show()

**⏸ Discuss.**
1. In the Task 3 table, $Z_1$ is almost exactly $1/(2w)$ once $w \gtrsim 1$. Explain why in one sentence.
2. Between $w = 1$ and $w = 1000$ the verdict went from favouring M1 to very strongly favouring M0. **Nothing about the data changed.** What did change, and why does the evidence integral care?
3. Why is there no such thing as an "uninformative" prior for model selection? (Hint: what happens to $1/(2w)$ as $w \to \infty$? And why can't you use an *improper* flat prior on $\theta$ at all?)
4. In Task 4 the three posteriors are identical while the three Bayes factors differ by a factor of 100. Finish the sentence: *"A diffuse prior is harmless for ___ and decisive for ___, because …"*
5. The Task 3 curve has a *minimum* near $w \approx 0.5$ and climbs back toward $BF_{01} = 1$ as $w \to 0$. What is M1 turning into as the box shrinks? Why is the BF most favourable to M1 for a box that is *just* wide enough to hold $\bar x$?
6. Compare with Part 2.3: as $n$ grows at a fixed p-value, the LRT's verdict never changes. What would happen to $Z_0$, and so to $BF_{01}$ for a fixed box?

*Your answers:* ...

**Task 5 (optional): from Bayes factor to posterior odds.** The Bayes factor updates whatever you believed before seeing the data:

$$\frac{p(M_1 \mid D)}{p(M_0 \mid D)} = \frac{Z_1}{Z_0} \times \frac{p(M_1)}{p(M_0)}.$$

Suppose you think $\theta = 0$ is 10× more plausible than a nonzero $\theta$ before the experiment. With the box $w = 1$, what are the posterior odds for M1, and the posterior probability of M1?

In [ ]:
prior_odds_10 = 1 / 10                      # p(M1) / p(M0)
bf10_w1 = ...                               # TODO: BF10 with the w = 1 box
post_odds_10 = ...                          # TODO
print(f"BF10 = {bf10_w1:.3f}   prior odds = {prior_odds_10:.2f}   posterior odds = {post_odds_10:.3f}   p(M1 | D) = {post_odds_10 / (1 + post_odds_10):.3f}")

### (Optional) Does the prior's *shape* matter, not just its width?

You only ever needed one function, `log_evidence_M1(xbar, se, prior)`, for any prior. So comparing shapes is just a matter of writing a second prior.

Compare two priors that are "equally wide" by one common measure, the standard deviation:

- a uniform box $\theta \sim U(-w, w)$, whose standard deviation is $w/\sqrt 3$,
- a Gaussian centred on 0 with that same standard deviation.

**⏸ Predict before running:** with the same standard deviation, will the two priors give the same Bayes factor? If not, which one will favour M1 more when the prior is much wider than the likelihood?

*Your prediction:* ...

In [ ]:
def gaussian_prior(sd):
    """Gaussian centred on 0 with standard deviation sd."""
    # TODO: return a function of theta, like uniform_prior does
    ...

# 1) Look first: two priors with the same standard deviation, against the likelihood
w_show = 3.0
th = np.linspace(-2 * w_show, 2 * w_show, 2000)
plt.plot(th, stats.norm.pdf(th, xbar, SE), color=BLUE, label="likelihood p(x̄ | θ)")
plt.plot(th, uniform_prior(w_show)(th), color=ORANGE, label=f"uniform, w = {w_show:g}")
plt.plot(th, gaussian_prior(w_show / np.sqrt(3))(th), color=AQUA, label=f"Gaussian, same SD = {w_show/np.sqrt(3):.2f}")
plt.axvline(xbar, color=MUTED, ls="--")
plt.yscale("log"); plt.ylim(1e-3, 5); plt.xlabel("θ"); plt.ylabel("density (log scale)"); plt.legend(); plt.show()

# 2) Then compute: log BF10 for both shapes as the width grows
log_Z0 = log_evidence_M0(xbar, SE)
widths = [0.1, 0.3, 1.0, 3.0, 10.0, 100.0]
print(f"{'w':>6} {'SD':>6} {'log BF10 uniform':>17} {'log BF10 Gaussian':>18} {'difference':>11}")
for w in widths:
    sd = w / np.sqrt(3)
    lb_u = ...        # TODO: log BF10 with uniform_prior(w)
    lb_g = ...        # TODO: log BF10 with gaussian_prior(sd)
    print(f"{w:6g} {sd:6.2f} {lb_u:17.3f} {lb_g:18.3f} {lb_g - lb_u:11.3f}")

ws_shape = np.logspace(-1.5, 2.5, 120)
plt.plot(ws_shape, [log_evidence_M1(xbar, SE, uniform_prior(w)) - log_Z0 for w in ws_shape], color=ORANGE, label="uniform")
plt.plot(ws_shape, [log_evidence_M1(xbar, SE, gaussian_prior(w / np.sqrt(3))) - log_Z0 for w in ws_shape], color=AQUA, label="Gaussian, same SD")
plt.axhline(0, color=INK, lw=1)
plt.xscale("log"); plt.xlabel("uniform half-width w  (Gaussian SD = w/√3)"); plt.ylabel("log BF₁₀"); plt.legend(); plt.show()

# 3) Your turn (optional): write any other prior as a function of theta and add it to the plot.
#    Ideas: a Cauchy (stats.cauchy.pdf), a box centred on xbar (uniform_prior(w, c=xbar)), a half-normal that forbids theta < 0.
#    Remember: it must be a normalized PDF.

**⏸ Discuss.**
1. For very narrow priors ($w = 0.1$) both shapes give almost the same answer, and $\log BF_{10}$ is close to 0. Why? What is M1 turning into?
2. For wide priors ($w \ge 10$) the difference settles to a *constant*. Look at the first plot: what is each prior's density at $\bar x$? Use that to predict the constant.
3. Two priors with the same standard deviation gave different evidence. Which property of the prior does the evidence actually care about?
4. *(Stretch)* Choose the Gaussian's width so that its density at $\bar x$ matches the box's $1/(2w)$ instead of matching the SD. Do the two Bayes factors now agree at large $w$?

*Your answers:* ...